In [1]:
# Imports
import pandas as pd 
import numpy as np 
import json 
from collections import deque
import random 
import sys 

sys.path.append('../utils')
from data_augment import * 

In [2]:
# HELPER methods
def format_symptoms(symptoms):
    return ', '.join(symptoms)

def format_precautions(precautions):
    return ', '.join(precautions)

def build_input(template, **kwargs):
    return template.format(**kwargs)

def build_output(template, **kwargs):
    return template.format(**kwargs)

def generate_prompt(input_txt):
    return f"{input_txt}"

In [3]:
input_templates = {
    "medical_qa": [
       "{question}",
       "I don’t really understand this: {question}",
       "can you explain this in simple terms? {question}",
       "i came across something and I’m confused: {question}",
       "What does this mean? {question}",
       "I’m trying to understand this better: {question}",
       "Could you break this down for me? {question}",
       "I saw this and it didn’t make sense to me: {question}"
    ],

    "symptom_diagnosis": [
        "i’ve been feeling {symptoms}. I’m not sure what’s going on.",
        "Lately I’ve had {symptoms} and I’m a bit worried",
        "something feels off—I’ve got {symptoms}.",
        "I’ve been dealing with {symptoms} for a few days",
        "I’m not feeling like myself… {symptoms}.",
        "i noticed {symptoms} recently and I’m concerned.",
        "I have had these symptoms: {symptoms}. Should I be worried?",
        "I’m unsure if this is serious, but I have {symptoms}.",
    ],

    "precaution_advice": [
        "what should I do if I have {symptoms}?",
        "I think I might have {disease}. What should I do",
        "how do I take care of myself if I’m experiencing {symptoms}?",
        "Is there anything I can do at home for {symptoms}?",
        "I’m not sure how serious this is—what steps should I take {symptoms}",
        "What should I avoid if I have {symptoms}?",
        "How do people usually manage {disease}?",
        "What’s the safest way to handle {symptoms}?"
    ],

    "patient_chat": [
        "{question}",
        "I’m really worried about this {question}",
        "I don’t know what to think about this. {question}",
        "Can you help me understand this better? {question}",
        "This has been stressing me out: {question}",
        "I keep thinking about this and I’m anxious: {question}",
        "Is this something serious? {question}",
        "I’m not sure if I should be concerned: {question}"
    ]
}


In [4]:
output_templates = {
    "medical_qa": [
        "{answer}",
        "In simple terms, {answer}",
        "What this means is {answer}",  
        "Let me explain it simply—{answer}",
        "Basically, {answer}",
        "Here’s the idea: {answer}",
        "To make this easier to understand, {answer}",
        "Put simply, {answer}"
    ],

    "symptom_diagnosis": [
        "From what you’re describing, this could possibly be related to {disease}, since it can present with symptoms like {symptoms}. That said, there are a few different possibilities depending on the full picture. How long have you been feeling this way?",
        "Symptoms like {symptoms} are sometimes seen with conditions such as {disease}. {description} It’s hard to be certain without more context, but this is one possibility. Have the symptoms been getting worse or staying about the same?",
        "I can see why this is concerning. One possible explanation is {disease}, which can sometimes cause symptoms like {symptoms}. However, other conditions may also look similar.",
        "There are a few things that could explain {symptoms}. One of them is {disease}, but it would depend on how these symptoms are evolving over time. Has anything made the symptoms better or worse?",
        "When symptoms like {symptoms} appear, one possibility we consider is {disease}. It’s important to look at duration and severity as well.",
        "This could potentially be related to {disease}. Have the symptoms in {symptoms} been getting worse or staying the same?",
        "I understand this feels worrying. Based on what you’ve shared, {disease} is one possible explanation, but many cases like this are not serious.",
        "One possible cause of {symptoms} is {disease}. It would help to know how long this has been going on and whether anything makes it better or worse."
    ],

    "precaution_advice": [
        "You can start by following {precautions}. These steps often help with symptoms like {symptoms}.",
        "{precautions} can help because they reduce strain on the body and support recovery from symptoms like {symptoms}.",
        "To manage this safely, it’s usually helpful to follow {precautions}. If things get worse, it’s best to get medical advice.",
        "For something like this, {precautions} are commonly recommended. They can help prevent symptoms from worsening.",
        "You can try {precautions} to help manage this, but if symptoms like {symptoms} continue or worsen, it’s important to get checked.",
        "These steps—{precautions}—help because they support recovery and reduce factors that can worsen {symptoms}.",
        "It might help to focus on {precautions} for now. Many people see improvement with these kinds of steps.",
        "A good starting point is {precautions}. This can often make symptoms like {symptoms} easier to manage."
    ],

    "patient_chat": [
        "Here's a simple way to think about it. {answer}",
        "I can help explain that. {answer}",
        "That's a good question. {answer} Do you want help understanding what to watch for next?",
        "Here is what that usually means. {answer}",
        "Let’s go through this together. {answer}",
        "Let's walk through it clearly. {answer} Is this something you're asking about yourself or for someone else?",
        "{answer} If you want, I can go into more detail on any part.",
        "You’re not alone in this. {answer}"
    ]
}

In [5]:
final_data = deque()

instructions = {

    "medical_qa": (
        "Explain the medical question clearly and accurately in simple, easy-to-understand language. "
        "Avoid jargon and focus on helping the patient understand."
    ),

    "symptom_diagnosis": (
        "Understand the symptoms and suggest possible conditions. "
        "Use reasoning, avoid certainty, and explain your thinking in a natural way. "
        "Acknowledge uncertainty and, when appropriate, ask a brief follow-up question."
    ),

    "precaution_advice": (
        "Provide practical and safe advice based on the situation. "
        "Explain why the precautions help and keep the tone calm and supportive. "
        "Mention when it may be important to seek medical care."
    ),

    "patient_chat": (
        "Respond like a calm and empathetic medical assistant. "
        "Acknowledge the user's feelings, be supportive, and explain things clearly in a natural, conversational way."
    )
}

In [6]:
# Augment data for symptom_diagnosis and precaution_advice 
augment()

/Users/rohan/Desktop/Python-Projects/NLP-Projects/AI-Clinician-Chatbot/notebooks/../utils/data_preprocess.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset_df[dataset_cols] = dataset_df[dataset_cols].apply(


In [7]:
# Load the datasets
with open('../JSON_data/symptom_dataset.json', 'r') as file:
    symptom_dataset = json.load(file)

medquad = pd.read_csv('../medquad_cleaned.csv')

In [8]:
len(set([
    tuple(sorted(sample['symptoms']))
    for sample in symptom_dataset
]))

3720

In [9]:
from collections import Counter

Counter([d['disease'] for d in symptom_dataset])

Counter({'dengue': 210,
         'migraine': 210,
         'chicken pox': 210,
         'hepatitis d': 210,
         'hepatitis e': 209,
         'common cold': 209,
         'hepatitis a': 209,
         'hepatitis b': 209,
         'typhoid': 209,
         'tuberculosis': 209,
         'diabetes': 209,
         'hyperthyroidism': 209,
         'pneumonia': 209,
         'hypoglycemia': 209,
         'jaundice': 209,
         'chronic cholestasis': 208,
         'varicose veins': 208,
         'hypothyroidism': 208,
         'alcoholic hepatitis': 208,
         'malaria': 208,
         '(vertigo) paroymsal  positional vertigo': 207,
         'bronchial asthma': 207,
         'hepatitis c': 207,
         'osteoarthristis': 207,
         'gerd': 207,
         'peptic ulcer disease': 207,
         'psoriasis': 207,
         'cervical spondylosis': 206,
         'impetigo': 206,
         'drug reaction': 206,
         'arthritis': 206,
         'hypertension': 206,
         'dimorphic hemo

In [10]:
def convert_symptoms(symptom_dataset):

    symptom_diagnosis_samples = 1
    precaution_advice_samples = 2

    for item in symptom_dataset:

        symptoms = format_symptoms(item['symptoms'])
        disease = item['disease']
        description = item['description']
        precautions = format_precautions(item['precaution'])

        for i in range(0, symptom_diagnosis_samples):
            data_dict = dict()

            idx = random.randint(0, len(input_templates['symptom_diagnosis']) - 1)

            input_template = input_templates['symptom_diagnosis'][idx]
            output_template = output_templates['symptom_diagnosis'][idx]

            input_txt = build_input(input_template, symptoms=symptoms)
            output_txt = build_output(output_template, disease=disease, symptoms=symptoms, description=description)


            data_dict['task_type'] = 'symptom_diagnosis'
            data_dict['instruction'] = instructions['symptom_diagnosis']
            data_dict['input'] = generate_prompt(input_txt)
            data_dict['output'] = output_txt

            final_data.append(data_dict)
        
        for i in range(0, precaution_advice_samples):
            data_dict = dict() 

            idx = random.randint(0, len(input_templates['precaution_advice']) - 1)

            input_template = input_templates['precaution_advice'][idx]
            output_template = output_templates['precaution_advice'][idx]

            input_txt = build_input(input_template, symptoms=symptoms, disease=disease)
            output_txt = build_output(output_template, precautions=precautions, symptoms=symptoms)

            data_dict['task_type'] = 'precaution_advice'
            data_dict['instruction'] = instructions['precaution_advice']
            data_dict['input'] = generate_prompt(input_txt)
            data_dict['output'] = output_txt

            final_data.append(data_dict) 



In [11]:
def convert_patient_qa(medquad):

    medical_qa_samples = 1
    patient_chat_samples = 2

    for _, row in medquad.iterrows():
        question = row['question']
        answer = row['answer']

        for i in range(0, medical_qa_samples): 
            data_dict = dict() 

            idx = random.randint(0, len(input_templates['medical_qa']) - 1)

            input_template = input_templates['medical_qa'][idx]
            output_template = output_templates['medical_qa'][idx]

            input_txt = build_input(input_template, question=question)
            output_txt = build_output(output_template, answer=answer)

            data_dict['task_type'] = 'medical_qa'
            data_dict['instruction'] = instructions['medical_qa']
            data_dict['input'] = generate_prompt(input_txt)
            data_dict['output'] = output_txt

            final_data.append(data_dict)

        for i in range(0, patient_chat_samples): 
            data_dict = dict() 

            idx = random.randint(0, len(input_templates['patient_chat']) - 1)

            input_template = input_templates['patient_chat'][idx]
            output_template = output_templates['patient_chat'][idx]

            input_txt = build_input(input_template, question=question)
            output_txt = build_output(output_template, answer=answer)

            data_dict['task_type'] = 'patient_chat'
            data_dict['instruction'] = instructions['patient_chat']
            data_dict['input'] = generate_prompt(input_txt)
            data_dict['output'] = output_txt

            final_data.append(data_dict)
        

        

        

In [12]:
convert_symptoms(symptom_dataset)
convert_patient_qa(medquad)

In [13]:
final_data = list(final_data)
final_dataset = pd.DataFrame(final_data)
final_dataset.head()

,task_type,instruction,input,output
0,symptom_diagnosis,Understand the symptoms and suggest possible c...,"i’ve been feeling itching, vomiting, yellowish...","From what you’re describing, this could possib..."
1,precaution_advice,Provide practical and safe advice based on the...,"What’s the safest way to handle itching, vomit...",A good starting point is take cool baths to he...
2,precaution_advice,Provide practical and safe advice based on the...,how do I take care of myself if I’m experienci...,"To manage this safely, it’s usually helpful to..."
3,symptom_diagnosis,Understand the symptoms and suggest possible c...,"I’m unsure if this is serious, but I have vomi...","One possible cause of vomiting, yellowish skin..."
4,precaution_advice,Provide practical and safe advice based on the...,how do I take care of myself if I’m experienci...,"To manage this safely, it’s usually helpful to..."


In [14]:
for sample in random.sample(final_data, 5):
    print(sample['input'])
    print(sample['output'])
    print("-----")

can you explain this in simple terms? what are the treatments for human t-cell leukemia virus type 2 ?
What this means is how might human t-cell leukemia virus, type 2 be treated? no cure or treatment exists for human t-cell leukemia virus, type 2 (htlv-2). management is focused on early detection and preventing the spread of htlv-2 to others. screening blood doners, promoting safe sex and discouraging needle sharing can decrease the number of new infections. mother-to-child transmission can be reduced by screening pregnant women so infected mothers can avoid breastfeeding.
-----
I’m not feeling like myself… fatigue, excessive hunger, irritability.
When symptoms like fatigue, excessive hunger, irritability appear, one possibility we consider is hypoglycemia. It’s important to look at duration and severity as well.
-----
i’ve been feeling fatigue, abdominal pain, joint pain, yellowing of eyes, loss of appetite, yellowish skin. I’m not sure what’s going on.
From what you’re describing, t

In [15]:
# Check for missing values
final_dataset.isnull().sum()

task_type      0
instruction    0
input          0
output         0
dtype: int64

In [16]:
final_dataset.drop_duplicates(inplace=True)

In [17]:
# Check the distribution of task types 
print(len(final_dataset[final_dataset['task_type'] == 'symptom_diagnosis']))
print(len(final_dataset[final_dataset['task_type'] == 'precaution_advice']))
print(len(final_dataset[final_dataset['task_type'] == 'medical_qa']))
print(len(final_dataset[final_dataset['task_type'] == 'patient_chat']))


8046
12772
9626
18052


In [18]:
final_dataset.to_csv('../final_dataset.csv')

In [20]:
final_dataset = final_dataset.drop(columns=['Unnamed: 0'], errors='ignore')

In [19]:
# Shuffle the dataset 
final_dataset = final_dataset.sample(frac=1, random_state=42).reset_index(drop=True)

In [21]:
train_size = int(0.8 * len(final_dataset))
val_size = int(0.1 * len(final_dataset))

train_dataset = final_dataset[:train_size]
val_dataset = final_dataset[train_size:train_size + val_size]
test_dataset = final_dataset[train_size + val_size:]

In [22]:
# Check the sizes of the splits
print(f"Train Size: {len(train_dataset)}")
print(f"Validation Size: {len(val_dataset)}")
print(f"Test Size: {len(test_dataset)}")

Train Size: 38796
Validation Size: 4849
Test Size: 4851


In [23]:
# Check distribution of task types in each split
print("Train Distribution:")
print(train_dataset['task_type'].value_counts())
print("-------------------------------------------------")

print("Validation Distribution:")
print(val_dataset['task_type'].value_counts())
print("-------------------------------------------------")

print("Test Distribution:")
print(test_dataset['task_type'].value_counts())

Train Distribution:
task_type
patient_chat         14474
precaution_advice    10223
medical_qa            7718
symptom_diagnosis     6381
Name: count, dtype: int64
-------------------------------------------------
Validation Distribution:
task_type
patient_chat         1804
precaution_advice    1274
medical_qa            922
symptom_diagnosis     849
Name: count, dtype: int64
-------------------------------------------------
Test Distribution:
task_type
patient_chat         1774
precaution_advice    1275
medical_qa            986
symptom_diagnosis     816
Name: count, dtype: int64


In [24]:
# Save the train, test, and validation data 
train_dataset.to_csv('../data/processed/train/train.csv', index=False)
val_dataset.to_csv('../data/processed/val/val.csv', index=False)
test_dataset.to_csv('../data/processed/test/test.csv', index=False)